# LAMPS CrewAI Full Live Pipeline on PyPI Packages

Runs the paper-style live pipeline:

`Fetcher -> Ollama Extractor -> CodeBERT Classifier -> Ollama Verdict`

Drive requirements:
- `NT230/data/d1/saved_models/checkpoint-best-acc/model.bin`
- optional package list: `NT230/data/live_packages.txt`
- Ollama key in `NT230/data/.env` as `$env:OLLAMA_API_KEY = "..."`, or set it manually in Colab.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, re, sys, json, shutil
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/My Drive/NT230')
DRIVE_DATA = DRIVE_ROOT / 'data'
MODEL_PATH = DRIVE_DATA / 'd1/saved_models/checkpoint-best-acc/model.bin'
PACKAGE_LIST_PATH = DRIVE_DATA / 'live_packages.txt'
OUTPUT_DIR = DRIVE_DATA / 'live_crewai_results'
LOCAL_MODEL = Path('/content/saved_models/checkpoint-best-acc/model.bin')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_MODEL.parent.mkdir(parents=True, exist_ok=True)
assert MODEL_PATH.exists(), f'Missing model.bin: {MODEL_PATH}'
shutil.copy(MODEL_PATH, LOCAL_MODEL)
print('Model:', LOCAL_MODEL, round(LOCAL_MODEL.stat().st_size / 1e6, 1), 'MB')
print('Package list:', PACKAGE_LIST_PATH if PACKAGE_LIST_PATH.exists() else 'default list')
print('Output:', OUTPUT_DIR)


In [ ]:
!pip install -q transformers==4.40.0 torch scikit-learn scipy pandas tqdm crewai ollama


In [ ]:
REPO_DIR = Path('/content/NT230')
if not REPO_DIR.exists():
    !git clone --depth=1 https://github.com/khoilv2005/NT230.git /content/NT230
sys.path.insert(0, str(REPO_DIR / 'src'))
print('Repo ready:', REPO_DIR)


In [ ]:
# Load OLLAMA_API_KEY from Drive .env if present.
env_path = DRIVE_DATA / '.env'
if env_path.exists():
    for line in env_path.read_text(encoding='utf-8', errors='ignore').splitlines():
        line = line.strip()
        m = re.match(r'^\$env:(\w+)\s*=\s*["\']?([^"\']+)["\']?', line)
        if m:
            os.environ.setdefault(m.group(1), m.group(2).strip())
        elif '=' in line and not line.startswith('#'):
            k, _, v = line.partition('=')
            os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))
print('OLLAMA_API_KEY:', 'SET' if os.getenv('OLLAMA_API_KEY') else 'NOT FOUND')
print('OLLAMA_MODEL:', os.getenv('OLLAMA_MODEL', 'deepseek-v4-flash:cloud'))

# Optional manual fallback:
# os.environ['OLLAMA_API_KEY'] = 'your-key-here'


In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU')


In [ ]:
from lamps.agents.classifier import ClassifierAgent
from lamps.agents.extractor import LLMArchiveExtractorAgent
from lamps.agents.fetcher import FetcherAgent
from lamps.agents.verdict import VerdictAgent
from lamps.crewai_pipeline import LampsCrewPipeline
from lamps.llms.ollama_client import OllamaClient

llm = OllamaClient(
    model=os.getenv('OLLAMA_MODEL', 'deepseek-v4-flash:cloud'),
    host='https://api.ollama.com',
)

pipeline = LampsCrewPipeline(
    fetcher=FetcherAgent(),
    extractor=LLMArchiveExtractorAgent(llm=llm),
    classifier=ClassifierAgent(checkpoint=str(LOCAL_MODEL), batch_size=64),
    verdict=VerdictAgent(llm=llm),
    crew_llm=None,
    verbose=False,
)
pipeline.crew = pipeline.build_crew()
print('CrewAI agents:', [a.role for a in pipeline.crew.agents])
print('CrewAI tasks:', len(pipeline.crew.tasks))


In [ ]:
def parse_package_line(line: str):
    line = line.strip()
    if not line or line.startswith('#'):
        return None
    if '==' in line:
        package, version = line.split('==', 1)
        return package.strip(), version.strip()
    return line, None

if PACKAGE_LIST_PATH.exists():
    package_specs = [parse_package_line(x) for x in PACKAGE_LIST_PATH.read_text(encoding='utf-8').splitlines()]
    package_specs = [x for x in package_specs if x]
else:
    package_specs = [
        ('requests', None),
        ('urllib3', None),
        ('certifi', None),
    ]

print('Packages to analyze:', len(package_specs))
print(package_specs[:20])


In [ ]:
from tqdm import tqdm

results = []
errors = []
out_jsonl = OUTPUT_DIR / 'package_results.jsonl'
out_errors = OUTPUT_DIR / 'errors.jsonl'

for package, version in tqdm(package_specs, desc='LAMPS CrewAI live'):
    try:
        result = pipeline.analyze_package(package, version=version)
        payload = result.to_dict()
        payload['crew_execution'] = pipeline.last_execution.to_dict() if pipeline.last_execution else None
        results.append(payload)
        with out_jsonl.open('a', encoding='utf-8') as f:
            f.write(json.dumps(payload, ensure_ascii=False) + '\n')
        print(package, '=>', payload['verdict']['label'], 'files=', payload['verdict']['n_files'])
    except Exception as exc:
        err = {'package': package, 'version': version, 'error': repr(exc)}
        errors.append(err)
        with out_errors.open('a', encoding='utf-8') as f:
            f.write(json.dumps(err, ensure_ascii=False) + '\n')
        print('ERROR', package, repr(exc))

summary = {
    'n_requested': len(package_specs),
    'n_success': len(results),
    'n_errors': len(errors),
    'results_path': str(out_jsonl),
    'errors_path': str(out_errors),
}
(OUTPUT_DIR / 'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(json.dumps(summary, indent=2))


In [ ]:
# Quick view
for payload in results[:10]:
    verdict = payload['verdict']
    print(payload['package'], verdict['label'], 'target=', verdict['target'], 'n_files=', verdict['n_files'])
